In [ ]:
import os
import json
import urllib.request
import urllib.error
import openai
from openai.types.chat import ChatCompletionMessage
from dotenv import load_dotenv
load_dotenv()  # .env 파일 자동 로드

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def _get(path: str):
    url = f"{BASE_URL}{path}"
    headers = {
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0 (compatible; AI-Client/1.0)",
    }
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = resp.read().decode("utf-8")
        return json.loads(data)
    except urllib.error.HTTPError as e:
        return {"error": f"HTTP {e.code}", "url": url}
    except urllib.error.URLError as e:
        return {"error": f"URL error: {e.reason}", "url": url}


def get_popular_movies():
    return json.dumps(_get("/movies"), ensure_ascii=False)


def get_movie_details(id):
    return json.dumps(_get(f"/movies/{id}"), ensure_ascii=False)


def get_movie_credits(id):
    return json.dumps(_get(f"/movies/{id}/credits"), ensure_ascii=False)


def get_similar_movies(id):
    return json.dumps(_get(f"/movies/{id}/similar"), ensure_ascii=False)


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "get_similar_movies": get_similar_movies,
}

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


In [ ]:
SYSTEM_PROMPT = """
너는 Movie Expert Agent이다.
사용자의 질문을 이해하고 아래 함수 중 하나를 선택해 호출하라.
필요할 때만 함수를 호출하고, 함수명과 인자를 정확히 지정하라.

사용 가능한 함수:
- get_popular_movies(): /movies에서 인기 영화 목록을 가져온다.
- get_movie_details(id): /movies/:id에서 영화 정보를 가져온다.
- get_movie_credits(id): /movies/:id/credits에서 출연진 및 제작진을 가져온다.
- get_similar_movies(id): /movies/:id/similar에서 유사한 영화 목록을 가져온다.
"""

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "인기 영화 목록을 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "영화 ID로 영화 상세 정보를 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "영화의 ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "영화 ID로 출연진 및 제작진 정보를 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "영화의 ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "영화 ID로 유사한 영화 목록을 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "영화의 ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

messages = [{"role": "system", "content": SYSTEM_PROMPT}]


In [ ]:
def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            if function_to_run is None:
                result = json.dumps({"error": f"Unknown tool: {function_name}"})
            else:
                result = function_to_run(**arguments)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )
        msg = response.choices[0].message

        process_ai_response(msg)
        if not msg.tool_calls:
            return


In [ ]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()
'''
User: 지금 인기 있는 영화 알려줘
[get_popular_movies() 호출]
AI: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Mercy**
   - 개봉일: 2026-01-20
   - 개요: 미래의 한 탐정이 아내를 살해한 혐의로 재판을 받고 있으며, 자신의 무죄를 입증하기 위해 고급 AI 판사에게 90분의 시간을 주어진다.
   - 평점: 7.12
   - ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **Shelter**
   - 개봉일: 2026-01-28
   - 개요: 외딴 섬에서 자가 망명 중인 한 남자가 폭풍으로부터 어린 소녀를 구하면서 벌어지는 사건들을 다룬다.
   - 평점: 7.0
   - ![Shelter](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

3. **The Orphans**
   - 개봉일: 2025-08-20
   - 개요: 고아원에서 친구였던 두 명이 첫사랑의 딸이 복수를 하려는 사건에 휘말리는 이야기.
   - 평점: 6.11
   - ![The Orphans](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **The Bluff**
   - 개봉일: 2026-02-17
   - 개요: 한 섬에서 평화로운 삶을 살아가던 여전사가 복수심에 불타는 옛 선장의 등장으로 가족을 지키기 위해 과거와 대면하게 된다.
   - 평점: 5.7
   - ![The Bluff](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

5. **A Woman Scorned**
   - 개봉일: 2025-06-09
   - 개요: 가족 주말을 보내던 두 자매가 지역 남성들에게 테러를 당하면서 복수를 결심하는 이야기.
   - 평점: 6.46
   - ![A Woman Scorned](https://image.tmdb.org/t/p/w780/dlOSBiNULMPzKIze84LDjvEN9z1.jpg)

User: 지금 인기 있는 영화 알려줘
[get_popular_movies() 호출]
AI: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Mercy**
   - 개봉일: 2026-01-20
   - 개요: 미래의 한 탐정이 아내를 살해한 혐의로 재판을 받고 있으며, 자신의 무죄를 입증하기 위해 고급 AI 판사에게 90분의 시간을 주어진다.
   - 평점: 7.12
   - ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **Shelter**
   - 개봉일: 2026-01-28
   - 개요: 외딴 섬에서 자가 망명 중인 한 남자가 폭풍으로부터 어린 소녀를 구하면서 벌어지는 사건들을 다룬다.
   - 평점: 7.0
   - ![Shelter](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

3. **The Orphans**
   - 개봉일: 2025-08-20
   - 개요: 고아원에서 친구였던 두 명이 첫사랑의 딸이 복수를 하려는 사건에 휘말리는 이야기.
   - 평점: 6.11
   - ![The Orphans](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **The Bluff**
   - 개봉일: 2026-02-17
   - 개요: 한 섬에서 평화로운 삶을 살아가던 여전사가 복수심에 불타는 옛 선장의 등장으로 가족을 지키기 위해 과거와 대면하게 된다.
   - 평점: 5.7
   - ![The Bluff](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

5. **A Woman Scorned**
   - 개봉일: 2025-06-09
   - 개요: 가족 주말을 보내던 두 자매가 지역 남성들에게 테러를 당하면서 복수를 결심하는 이야기.
   - 평점: 6.46
   - ![A Woman Scorned](https://image.tmdb.org/t/p/w780/dlOSBiNULMPzKIze84LDjvEN9z1.jpg)

더 많은 영화 정보를 원하시면 말씀해 주세요!
User: 지금 인기 있는 영화 알려줘
[get_popular_movies() 호출]
AI: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Mercy**
   - 개봉일: 2026-01-20
   - 개요: 미래의 한 탐정이 아내를 살해한 혐의로 재판을 받고 있으며, 자신의 무죄를 입증하기 위해 고급 AI 판사에게 90분의 시간을 주어진다.
   - 평점: 7.12
   - ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **Shelter**
   - 개봉일: 2026-01-28
   - 개요: 외딴 섬에서 자가 망명 중인 한 남자가 폭풍으로부터 어린 소녀를 구하면서 벌어지는 사건들을 다룬다.
   - 평점: 7.0
   - ![Shelter](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

3. **The Orphans**
   - 개봉일: 2025-08-20
   - 개요: 고아원에서 친구였던 두 명이 첫사랑의 딸이 복수를 하려는 사건에 휘말리는 이야기.
   - 평점: 6.11
   - ![The Orphans](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **The Bluff**
   - 개봉일: 2026-02-17
   - 개요: 한 섬에서 평화로운 삶을 살아가던 여전사가 복수심에 불타는 옛 선장의 등장으로 가족을 지키기 위해 과거와 대면하게 된다.
   - 평점: 5.7
   - ![The Bluff](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

5. **A Woman Scorned**
   - 개봉일: 2025-06-09
   - 개요: 가족 주말을 보내던 두 자매가 지역 남성들에게 테러를 당하면서 복수를 결심하는 이야기.
   - 평점: 6.46
   - ![A Woman Scorned](https://image.tmdb.org/t/p/w780/dlOSBiNULMPzKIze84LDjvEN9z1.jpg)

더 많은 영화 정보를 원하시면 말씀해 주세요!
User: The Bluff에 대해서 더 알려줘
User: 지금 인기 있는 영화 알려줘
[get_popular_movies() 호출]
AI: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Mercy**
   - 개봉일: 2026-01-20
   - 개요: 미래의 한 탐정이 아내를 살해한 혐의로 재판을 받고 있으며, 자신의 무죄를 입증하기 위해 고급 AI 판사에게 90분의 시간을 주어진다.
   - 평점: 7.12
   - ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **Shelter**
   - 개봉일: 2026-01-28
   - 개요: 외딴 섬에서 자가 망명 중인 한 남자가 폭풍으로부터 어린 소녀를 구하면서 벌어지는 사건들을 다룬다.
   - 평점: 7.0
   - ![Shelter](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

3. **The Orphans**
   - 개봉일: 2025-08-20
   - 개요: 고아원에서 친구였던 두 명이 첫사랑의 딸이 복수를 하려는 사건에 휘말리는 이야기.
   - 평점: 6.11
   - ![The Orphans](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **The Bluff**
   - 개봉일: 2026-02-17
   - 개요: 한 섬에서 평화로운 삶을 살아가던 여전사가 복수심에 불타는 옛 선장의 등장으로 가족을 지키기 위해 과거와 대면하게 된다.
   - 평점: 5.7
   - ![The Bluff](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

5. **A Woman Scorned**
   - 개봉일: 2025-06-09
   - 개요: 가족 주말을 보내던 두 자매가 지역 남성들에게 테러를 당하면서 복수를 결심하는 이야기.
   - 평점: 6.46
   - ![A Woman Scorned](https://image.tmdb.org/t/p/w780/dlOSBiNULMPzKIze84LDjvEN9z1.jpg)

더 많은 영화 정보를 원하시면 말씀해 주세요!
User: The Bluff에 대해서 더 알려줘
[get_movie_details(799882) 호출]
AI: **The Bluff**에 대한 정보는 다음과 같습니다:

- **제목**: The Bluff
- **개봉일**: 2026-02-17
- **장르**: 액션, 스릴러
- **상영 시간**: 102분
- **언어**: 영어
- **태그라인**: Betrayal cuts deep.
- **평점**: 5.73 (투표 수: 45)
- **개요**: 한 외딴 섬에서의 평화로운 삶이 복수심에 불타는 이전 선장의 귀환으로 산산조각 나고, 숙련된 전직 해적이 가족을 지키기 위해 자신의 피비린내 나는 과거에 맞서 싸우는 이야기입니다.
- **제작사**:
    - AGBO
    - Cinestar Pictures
    - Big Indie Pictures
- **홈페이지**: [The Bluff on Amazon](https://www.amazon.com/gp/video/detail/B0G569YVZW)

![The Bluff Poster](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

User: 지금 인기 있는 영화 알려줘
[get_popular_movies() 호출]
AI: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Mercy**
   - 개봉일: 2026-01-20
   - 개요: 미래의 한 탐정이 아내를 살해한 혐의로 재판을 받고 있으며, 자신의 무죄를 입증하기 위해 고급 AI 판사에게 90분의 시간을 주어진다.
   - 평점: 7.12
   - ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **Shelter**
   - 개봉일: 2026-01-28
   - 개요: 외딴 섬에서 자가 망명 중인 한 남자가 폭풍으로부터 어린 소녀를 구하면서 벌어지는 사건들을 다룬다.
   - 평점: 7.0
   - ![Shelter](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

3. **The Orphans**
   - 개봉일: 2025-08-20
   - 개요: 고아원에서 친구였던 두 명이 첫사랑의 딸이 복수를 하려는 사건에 휘말리는 이야기.
   - 평점: 6.11
   - ![The Orphans](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **The Bluff**
   - 개봉일: 2026-02-17
   - 개요: 한 섬에서 평화로운 삶을 살아가던 여전사가 복수심에 불타는 옛 선장의 등장으로 가족을 지키기 위해 과거와 대면하게 된다.
   - 평점: 5.7
   - ![The Bluff](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

5. **A Woman Scorned**
   - 개봉일: 2025-06-09
   - 개요: 가족 주말을 보내던 두 자매가 지역 남성들에게 테러를 당하면서 복수를 결심하는 이야기.
   - 평점: 6.46
   - ![A Woman Scorned](https://image.tmdb.org/t/p/w780/dlOSBiNULMPzKIze84LDjvEN9z1.jpg)

더 많은 영화 정보를 원하시면 말씀해 주세요!
User: The Bluff에 대해서 더 알려줘
[get_movie_details(799882) 호출]
AI: **The Bluff**에 대한 정보는 다음과 같습니다:

- **제목**: The Bluff
- **개봉일**: 2026-02-17
- **장르**: 액션, 스릴러
- **상영 시간**: 102분
- **언어**: 영어
- **태그라인**: Betrayal cuts deep.
- **평점**: 5.73 (투표 수: 45)
- **개요**: 한 외딴 섬에서의 평화로운 삶이 복수심에 불타는 이전 선장의 귀환으로 산산조각 나고, 숙련된 전직 해적이 가족을 지키기 위해 자신의 피비린내 나는 과거에 맞서 싸우는 이야기입니다.
- **제작사**:
    - AGBO
    - Cinestar Pictures
    - Big Indie Pictures
- **홈페이지**: [The Bluff on Amazon](https://www.amazon.com/gp/video/detail/B0G569YVZW)

![The Bluff Poster](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

더 궁금한 사항이 있으면 말씀해 주세요!
User: 지금 인기 있는 영화 알려줘
[get_popular_movies() 호출]
AI: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Mercy**
   - 개봉일: 2026-01-20
   - 개요: 미래의 한 탐정이 아내를 살해한 혐의로 재판을 받고 있으며, 자신의 무죄를 입증하기 위해 고급 AI 판사에게 90분의 시간을 주어진다.
   - 평점: 7.12
   - ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **Shelter**
   - 개봉일: 2026-01-28
   - 개요: 외딴 섬에서 자가 망명 중인 한 남자가 폭풍으로부터 어린 소녀를 구하면서 벌어지는 사건들을 다룬다.
   - 평점: 7.0
   - ![Shelter](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

3. **The Orphans**
   - 개봉일: 2025-08-20
   - 개요: 고아원에서 친구였던 두 명이 첫사랑의 딸이 복수를 하려는 사건에 휘말리는 이야기.
   - 평점: 6.11
   - ![The Orphans](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **The Bluff**
   - 개봉일: 2026-02-17
   - 개요: 한 섬에서 평화로운 삶을 살아가던 여전사가 복수심에 불타는 옛 선장의 등장으로 가족을 지키기 위해 과거와 대면하게 된다.
   - 평점: 5.7
   - ![The Bluff](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

5. **A Woman Scorned**
   - 개봉일: 2025-06-09
   - 개요: 가족 주말을 보내던 두 자매가 지역 남성들에게 테러를 당하면서 복수를 결심하는 이야기.
   - 평점: 6.46
   - ![A Woman Scorned](https://image.tmdb.org/t/p/w780/dlOSBiNULMPzKIze84LDjvEN9z1.jpg)

더 많은 영화 정보를 원하시면 말씀해 주세요!
User: The Bluff에 대해서 더 알려줘
[get_movie_details(799882) 호출]
AI: **The Bluff**에 대한 정보는 다음과 같습니다:

- **제목**: The Bluff
- **개봉일**: 2026-02-17
- **장르**: 액션, 스릴러
- **상영 시간**: 102분
- **언어**: 영어
- **태그라인**: Betrayal cuts deep.
- **평점**: 5.73 (투표 수: 45)
- **개요**: 한 외딴 섬에서의 평화로운 삶이 복수심에 불타는 이전 선장의 귀환으로 산산조각 나고, 숙련된 전직 해적이 가족을 지키기 위해 자신의 피비린내 나는 과거에 맞서 싸우는 이야기입니다.
- **제작사**:
    - AGBO
    - Cinestar Pictures
    - Big Indie Pictures
- **홈페이지**: [The Bluff on Amazon](https://www.amazon.com/gp/video/detail/B0G569YVZW)

![The Bluff Poster](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

더 궁금한 사항이 있으면 말씀해 주세요!
User: 비슷한 영화 추천해죠
User: 지금 인기 있는 영화 알려줘
[get_popular_movies() 호출]
AI: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Mercy**
   - 개봉일: 2026-01-20
   - 개요: 미래의 한 탐정이 아내를 살해한 혐의로 재판을 받고 있으며, 자신의 무죄를 입증하기 위해 고급 AI 판사에게 90분의 시간을 주어진다.
   - 평점: 7.12
   - ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **Shelter**
   - 개봉일: 2026-01-28
   - 개요: 외딴 섬에서 자가 망명 중인 한 남자가 폭풍으로부터 어린 소녀를 구하면서 벌어지는 사건들을 다룬다.
   - 평점: 7.0
   - ![Shelter](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

3. **The Orphans**
   - 개봉일: 2025-08-20
   - 개요: 고아원에서 친구였던 두 명이 첫사랑의 딸이 복수를 하려는 사건에 휘말리는 이야기.
   - 평점: 6.11
   - ![The Orphans](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **The Bluff**
   - 개봉일: 2026-02-17
   - 개요: 한 섬에서 평화로운 삶을 살아가던 여전사가 복수심에 불타는 옛 선장의 등장으로 가족을 지키기 위해 과거와 대면하게 된다.
   - 평점: 5.7
   - ![The Bluff](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

5. **A Woman Scorned**
   - 개봉일: 2025-06-09
   - 개요: 가족 주말을 보내던 두 자매가 지역 남성들에게 테러를 당하면서 복수를 결심하는 이야기.
   - 평점: 6.46
   - ![A Woman Scorned](https://image.tmdb.org/t/p/w780/dlOSBiNULMPzKIze84LDjvEN9z1.jpg)

더 많은 영화 정보를 원하시면 말씀해 주세요!
User: The Bluff에 대해서 더 알려줘
[get_movie_details(799882) 호출]
AI: **The Bluff**에 대한 정보는 다음과 같습니다:

- **제목**: The Bluff
- **개봉일**: 2026-02-17
- **장르**: 액션, 스릴러
- **상영 시간**: 102분
- **언어**: 영어
- **태그라인**: Betrayal cuts deep.
- **평점**: 5.73 (투표 수: 45)
- **개요**: 한 외딴 섬에서의 평화로운 삶이 복수심에 불타는 이전 선장의 귀환으로 산산조각 나고, 숙련된 전직 해적이 가족을 지키기 위해 자신의 피비린내 나는 과거에 맞서 싸우는 이야기입니다.
- **제작사**:
    - AGBO
    - Cinestar Pictures
    - Big Indie Pictures
- **홈페이지**: [The Bluff on Amazon](https://www.amazon.com/gp/video/detail/B0G569YVZW)

![The Bluff Poster](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

더 궁금한 사항이 있으면 말씀해 주세요!
User: 비슷한 영화 추천해죠
[get_similar_movies(799882) 호출]
AI: **The Bluff**와 비슷한 영화 추천 목록입니다:

1. **The Naked Prey**
   - 개봉일: 1965-06-01
   - 개요: 사파리 여행 중, 선물을 거부한 한 남자가 부족에게 붙잡혀 나머지 멤버들이 끔찍히 죽어가는 상황에서 생존자가 되어 사투를 벌인다.
   - 평점: 6.6
   - ![The Naked Prey](https://image.tmdb.org/t/p/w780/lBm8f2eVBKUKVqrdyJTNwdfAUqi.jpg)

2. **Island Song**
   - 개봉일: 2013-01-01
   - 개요: 로스앤젤레스의 유명 라디오 DJ가 섬에 갇히고 신비로운 인물과 범죄 세력의 싸움에 휘말리게 되는 이야기.
   - 평점: 정보 없음
   - ![Island Song](http://via.placeholder.com/780x400) *(포스터 없음)*

3. **Captain Clegg**
   - 개봉일: 1962-06-13
   - 개요: 해적들이 coastal town을 테러하는 "Marsh Phantoms"를 조사하는 이야기.
   - 평점: 6.27
   - ![Captain Clegg](https://image.tmdb.org/t/p/w780/5wQbnaBidXchb2WSSCdEFgX0ZyT.jpg)

4. **The Emperor of Paris**
   - 개봉일: 2018-12-19
   - 개요: 전설적인 죄수 프랑수아 비도크가 본래의 비극적인 과거에서 벗어나려는 이야기를 다룬다.
   - 평점: 6.3
   - ![The Emperor of Paris](https://image.tmdb.org/t/p/w780/7mZBfj3dFIzQgMVfkDwi9u4JBxI.jpg)

5. **Tombstone**
   - 개봉일: 1993-12-25
   - 개요: 전설적인 경찰관 와이어트 어프가 형제들과 함께 범죄자들에게 맞서 싸우는 이야기.
   - 평점: 7.58
   - ![Tombstone](https://image.tmdb.org/t/p/w780/wGFCvylul8iEQhJOKfwZGGvXMzA.jpg)

원하는 영화에 대한 추가 정보가 필요하시거나 더 많은 추천이 필요하시다면 말씀해 주세요!
'''